# Final Stage F 51R2 versus 53R2 call-input comparator

This notebook contains: (1) a capture cell to run immediately before the original build cell in Notebook 51R2 or Notebook 53R2; (2) a finalisation cell; and (3) the comparator. Capture both sides in fresh reconstructions, then run the comparator.

The comparator pairs calls by the semantic Boolean mask hash, so historical `M5` versus `M5_REMOVE` naming does not affect pairing. It tests observed input, exact mask, radius, OpenCV method and raw result separately.


## A. Capture setup
Run this in the original notebook immediately before its original reconstruction/build cell. Set `CAPTURE_SIDE` appropriately.


In [ ]:
# ============================================================
# CAPTURE SETUP - RUN IN THE ORIGINAL NOTEBOOK IMMEDIATELY
# BEFORE ITS ORIGINAL CANDIDATE/PAGE-BUILD CELL
#
# Set CAPTURE_SIDE to "51R2" in Notebook 51R2 and "53R2" in
# Notebook 53R2. Use a fresh output/rebuild so cv2.inpaint is
# actually called rather than checkpoints being reused.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib, inspect, json, platform, sys
import cv2
import numpy as np
import pandas as pd

CAPTURE_SIDE = '51R2'  # CHANGE TO '53R2' IN NOTEBOOK 53R2
PROJECT_ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
CAPTURE_SESSION = 'stage_f_final_call_inputs'

CAPTURE_ROOT = PROJECT_ROOT / CAPTURE_SESSION / CAPTURE_SIDE
CALL_ROOT = CAPTURE_ROOT / 'calls'
CALL_ROOT.mkdir(parents=True, exist_ok=True)

# Clean only this side's previous temporary capture. This does not
# touch candidate pages or historical runs.
for p in sorted(CALL_ROOT.glob('call_*')):
    if p.is_dir():
        import shutil
        shutil.rmtree(p)


def semantic_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def infer_context():
    out = {}
    names = (
        'page', 'page_id', 'page_name', 'region_id', 'region', 'rid',
        'arm', 'restorer', 'rest', 'method', 'regional', 'regional_m5',
        'm5', 'mask', 'observed', 'obs'
    )
    frames = inspect.stack()
    try:
        for fi in frames[2:25]:
            loc = fi.frame.f_locals
            for name in names:
                if name not in loc or name in out:
                    continue
                value = loc[name]
                if isinstance(value, (str, int, float, bool, Path, np.integer, np.floating, np.bool_)):
                    out[name] = str(value)
                elif isinstance(value, np.ndarray):
                    out[name + '_shape'] = list(value.shape)
                    out[name + '_dtype'] = str(value.dtype)
                    out[name + '_sha256'] = semantic_hash(value)
                    if value.ndim == 2:
                        out[name + '_nonzero'] = int((value != 0).sum())
    finally:
        del frames
    return out


if hasattr(cv2, '_stage_f_original_inpaint'):
    _ORIGINAL_INPAINT = cv2._stage_f_original_inpaint
else:
    _ORIGINAL_INPAINT = cv2.inpaint
    cv2._stage_f_original_inpaint = _ORIGINAL_INPAINT

_CAPTURE_ROWS = []
_CALL_NUMBER = 0


def _captured_inpaint(source, mask, radius, flags):
    global _CALL_NUMBER
    _CALL_NUMBER += 1
    source_before = np.ascontiguousarray(np.asarray(source).copy())
    mask_before = np.ascontiguousarray(np.asarray(mask).copy())
    result = _ORIGINAL_INPAINT(source, mask, radius, flags)
    result_array = np.ascontiguousarray(np.asarray(result).copy())
    context = infer_context()

    call_dir = CALL_ROOT / f'call_{_CALL_NUMBER:06d}'
    call_dir.mkdir(parents=True, exist_ok=False)
    np.save(call_dir / 'source.npy', source_before, allow_pickle=False)
    np.save(call_dir / 'mask.npy', mask_before, allow_pickle=False)
    np.save(call_dir / 'result.npy', result_array, allow_pickle=False)

    page = context.get('page') or context.get('page_id') or context.get('page_name')
    region = context.get('region_id') or context.get('region') or context.get('rid')
    method_name = 'INPAINT_TELEA' if int(flags) == int(cv2.INPAINT_TELEA) else str(int(flags))
    row = {
        'side': CAPTURE_SIDE,
        'call_number': _CALL_NUMBER,
        'page': page,
        'region_id': region,
        'radius': float(radius),
        'flags': int(flags),
        'flags_name': method_name,
        'source_shape': list(source_before.shape),
        'source_dtype': str(source_before.dtype),
        'source_sha256': semantic_hash(source_before),
        'mask_shape': list(mask_before.shape),
        'mask_dtype': str(mask_before.dtype),
        'mask_sha256': semantic_hash(mask_before),
        'mask_boolean_sha256': semantic_hash((mask_before > 0).astype(np.uint8)),
        'mask_pixels': int((mask_before > 0).sum()),
        'result_sha256': semantic_hash(result_array),
        'call_directory': str(call_dir),
        'context': context,
    }
    (call_dir / 'metadata.json').write_text(json.dumps(row, indent=2, default=str))
    _CAPTURE_ROWS.append(row)
    print('CAPTURED', CAPTURE_SIDE, _CALL_NUMBER, page, region, method_name, 'mask_pixels=', row['mask_pixels'])
    return result


cv2.inpaint = _captured_inpaint

(CAPTURE_ROOT / 'capture_environment.json').write_text(json.dumps({
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'side': CAPTURE_SIDE,
    'python': sys.version,
    'opencv': cv2.__version__,
    'numpy': np.__version__,
    'platform': platform.platform(),
}, indent=2))

print('Capture installed for:', CAPTURE_SIDE)
print('Capture root:', CAPTURE_ROOT)
print('NOW RUN THE ORIGINAL BUILD CELL UNCHANGED.')


## B. Capture finalisation
Run this after the original reconstruction/build cell completes.


In [ ]:
# ============================================================
# CAPTURE FINALISATION - RUN AFTER THE ORIGINAL BUILD CELL
# ============================================================

if '_ORIGINAL_INPAINT' not in globals():
    raise RuntimeError('Run the capture setup cell first.')

cv2.inpaint = _ORIGINAL_INPAINT

rows = []
for item in _CAPTURE_ROWS:
    flat = {k: v for k, v in item.items() if k != 'context'}
    for k, v in item.get('context', {}).items():
        flat[f'context_{k}'] = v
    rows.append(flat)

capture_df = pd.DataFrame(rows)
if capture_df.empty:
    raise RuntimeError(
        'No cv2.inpaint calls were captured. The original build probably '
        'reused checkpoints. Force a fresh reconstruction and rerun.'
    )

capture_df.to_csv(CAPTURE_ROOT / 'captured_calls.csv', index=False)
(CAPTURE_ROOT / 'capture_complete.json').write_text(json.dumps({
    'side': CAPTURE_SIDE,
    'calls': int(len(capture_df)),
    'telea_calls': int((capture_df.flags_name == 'INPAINT_TELEA').sum()),
    'complete': True,
}, indent=2))

print('Capture complete:', CAPTURE_ROOT)
display(capture_df[['call_number','page','region_id','flags_name','source_shape','mask_pixels']])


## C. Compare completed captures
Run after both `51R2` and `53R2` capture folders contain `capture_complete.json`.


In [ ]:
# ============================================================
# FINAL 51R2-versus-53R2 CALL-INPUT COMPARATOR
#
# Run after both capture directories have been completed.
# It pairs calls primarily by page + exact Boolean-mask identity.
# If region IDs are available on both sides, they are also reported.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys
import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
CAPTURE_BASE = ROOT / 'stage_f_final_call_inputs'
LEFT_ROOT = CAPTURE_BASE / '51R2'
RIGHT_ROOT = CAPTURE_BASE / '53R2'
OUT = CAPTURE_BASE / ('comparison_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
TABLES = OUT / 'tables'
DIFFS = OUT / 'differences'
for p in [OUT, TABLES, DIFFS]:
    p.mkdir(parents=True, exist_ok=False)

FAILING_PAGES = {'Page003','Page004','Page005','Page008','Page036','Page040','Page049'}
EXPECTED_TOTAL_RESIDUAL = 240


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def semantic_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()


def load_side(root, side):
    require((root / 'capture_complete.json').exists(), f'{side} capture is incomplete: {root}')
    rows = []
    for call_dir in sorted((root / 'calls').glob('call_*')):
        meta_path = call_dir / 'metadata.json'
        require(meta_path.exists(), f'Missing metadata: {meta_path}')
        meta = json.loads(meta_path.read_text())
        source = np.load(call_dir / 'source.npy', allow_pickle=False)
        mask = np.load(call_dir / 'mask.npy', allow_pickle=False)
        result = np.load(call_dir / 'result.npy', allow_pickle=False)
        require(semantic_hash(source) == meta['source_sha256'], f'{side}: source hash failure {call_dir}')
        require(semantic_hash(mask) == meta['mask_sha256'], f'{side}: mask hash failure {call_dir}')
        require(semantic_hash(result) == meta['result_sha256'], f'{side}: result hash failure {call_dir}')
        rows.append({'meta': meta, 'source': source, 'mask': mask, 'result': result, 'call_dir': call_dir})
    require(rows, f'No captured calls for {side}')
    return rows


left = load_side(LEFT_ROOT, '51R2')
right = load_side(RIGHT_ROOT, '53R2')

# Retain Telea calls. When page metadata is available, prioritise the seven
# known failing pages. Calls without page metadata remain eligible for
# hash-based pairing rather than being silently discarded.
def telea_calls(rows):
    return [r for r in rows if r['meta'].get('flags_name') == 'INPAINT_TELEA']

left = telea_calls(left)
right = telea_calls(right)
require(left and right, 'Both captures must contain INPAINT_TELEA calls.')

# Index 53R2 calls by exact Boolean mask hash. This is the safest pairing key
# because historical M5/M5_REMOVE naming may differ while the semantic mask
# is the object that controls cv2.inpaint.
right_by_mask = {}
for r in right:
    key = r['meta']['mask_boolean_sha256']
    right_by_mask.setdefault(key, []).append(r)

pairs = []
unmatched_left = []
used_right = set()

for l in left:
    key = l['meta']['mask_boolean_sha256']
    candidates = [r for r in right_by_mask.get(key, []) if str(r['call_dir']) not in used_right]

    # Prefer the same page if both sides recovered page metadata.
    lp = l['meta'].get('page')
    same_page = [r for r in candidates if lp and r['meta'].get('page') == lp]
    if len(same_page) == 1:
        candidates = same_page

    if len(candidates) == 1:
        r = candidates[0]
        used_right.add(str(r['call_dir']))
        pairs.append((l, r, 'EXACT_BOOLEAN_MASK'))
    else:
        unmatched_left.append({
            'left_call': str(l['call_dir']),
            'page': lp,
            'region_id': l['meta'].get('region_id'),
            'mask_boolean_sha256': key,
            'candidate_count': len(candidates),
        })

unmatched_right = [
    {
        'right_call': str(r['call_dir']),
        'page': r['meta'].get('page'),
        'region_id': r['meta'].get('region_id'),
        'mask_boolean_sha256': r['meta']['mask_boolean_sha256'],
    }
    for r in right
    if str(r['call_dir']) not in used_right
]

comparison_rows = []
coordinate_rows = []

for l, r, pairing_method in pairs:
    ls, rs = l['source'], r['source']
    lm, rm = l['mask'] > 0, r['mask'] > 0
    lr, rr = l['result'], r['result']

    same_shape = ls.shape == rs.shape == lm.shape == rm.shape == lr.shape == rr.shape
    if not same_shape:
        comparison_rows.append({
            'page_51': l['meta'].get('page'),
            'page_53': r['meta'].get('page'),
            'region_51': l['meta'].get('region_id'),
            'region_53': r['meta'].get('region_id'),
            'pairing_method': pairing_method,
            'shape_exact': False,
            'source_exact': False,
            'mask_exact': False,
            'radius_exact': l['meta']['radius'] == r['meta']['radius'],
            'flags_exact': l['meta']['flags'] == r['meta']['flags'],
            'result_exact': False,
            'diagnosis': 'SHAPE_DIFFERENCE',
        })
        continue

    source_diff = ls != rs
    mask_xor = np.logical_xor(lm, rm)
    result_diff = lr != rr
    inside_union = lm | rm

    source_exact = not source_diff.any()
    mask_exact = not mask_xor.any()
    radius_exact = float(l['meta']['radius']) == float(r['meta']['radius'])
    flags_exact = int(l['meta']['flags']) == int(r['meta']['flags'])
    result_exact = not result_diff.any()

    if not source_exact:
        diagnosis = 'OBSERVED_INPUT_DIFFERS'
        first_correction = 'Make 53R2 load the same observed array/path as fresh 51R2.'
    elif not mask_exact:
        diagnosis = 'REGIONAL_MASK_DIFFERS'
        first_correction = 'Make 53R2 construct the regional M5 mask exactly as fresh 51R2.'
    elif not radius_exact or not flags_exact:
        diagnosis = 'INPAINT_PARAMETERS_DIFFER'
        first_correction = 'Make 53R2 use the same radius and OpenCV inpaint flag as 51R2.'
    elif not result_exact:
        diagnosis = 'IDENTICAL_INPUTS_BUT_RAW_RESULT_DIFFERS'
        first_correction = 'Compare software/runtime and source/result memory layout; logic inputs already match.'
    else:
        diagnosis = 'CALL_EXACT'
        first_correction = 'No call-level correction required; inspect later composition/order.'

    page_label = l['meta'].get('page') or r['meta'].get('page') or 'UNKNOWN_PAGE'
    region_label = l['meta'].get('region_id') or r['meta'].get('region_id') or Path(l['call_dir']).name
    safe_name = (str(page_label) + '__' + str(region_label)).replace('/','_').replace('\\','_').replace(':','_').replace(' ','_')
    d = DIFFS / safe_name
    d.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(d / 'source_absolute_difference.png'), np.abs(ls.astype(np.int16)-rs.astype(np.int16)).astype(np.uint8))
    cv2.imwrite(str(d / 'mask_xor.png'), mask_xor.astype(np.uint8)*255)
    cv2.imwrite(str(d / 'result_absolute_difference.png'), np.abs(lr.astype(np.int16)-rr.astype(np.int16)).astype(np.uint8))

    comparison_rows.append({
        'page_51': l['meta'].get('page'),
        'page_53': r['meta'].get('page'),
        'region_51': l['meta'].get('region_id'),
        'region_53': r['meta'].get('region_id'),
        'call_51': str(l['call_dir']),
        'call_53': str(r['call_dir']),
        'pairing_method': pairing_method,
        'shape_exact': True,
        'source_exact': source_exact,
        'source_different_pixels': int(source_diff.sum()),
        'mask_exact': mask_exact,
        'mask_xor_pixels': int(mask_xor.sum()),
        'mask_pixels_51': int(lm.sum()),
        'mask_pixels_53': int(rm.sum()),
        'radius_51': float(l['meta']['radius']),
        'radius_53': float(r['meta']['radius']),
        'radius_exact': radius_exact,
        'flags_51': int(l['meta']['flags']),
        'flags_53': int(r['meta']['flags']),
        'flags_exact': flags_exact,
        'result_exact': result_exact,
        'result_different_pixels': int(result_diff.sum()),
        'result_different_inside_mask_union': int((result_diff & inside_union).sum()),
        'result_max_absolute_difference': int(np.abs(lr.astype(np.int16)-rr.astype(np.int16)).max()),
        'diagnosis': diagnosis,
        'first_correction': first_correction,
        'source_sha256_51': semantic_hash(ls),
        'source_sha256_53': semantic_hash(rs),
        'mask_boolean_sha256_51': semantic_hash(lm.astype(np.uint8)),
        'mask_boolean_sha256_53': semantic_hash(rm.astype(np.uint8)),
        'result_sha256_51': semantic_hash(lr),
        'result_sha256_53': semantic_hash(rr),
    })

    ys, xs = np.where(source_diff | mask_xor | result_diff)
    for y, x in zip(ys, xs):
        coordinate_rows.append({
            'page': page_label,
            'region': region_label,
            'y': int(y),
            'x': int(x),
            'source_51': int(ls[y,x]),
            'source_53': int(rs[y,x]),
            'mask_51': bool(lm[y,x]),
            'mask_53': bool(rm[y,x]),
            'result_51': int(lr[y,x]),
            'result_53': int(rr[y,x]),
        })

comparison_df = pd.DataFrame(comparison_rows)
coordinate_df = pd.DataFrame(coordinate_rows, columns=['page','region','y','x','source_51','source_53','mask_51','mask_53','result_51','result_53'])
unmatched_left_df = pd.DataFrame(unmatched_left)
unmatched_right_df = pd.DataFrame(unmatched_right)

comparison_df.to_csv(TABLES / 'call_input_comparison.csv', index=False)
coordinate_df.to_csv(TABLES / 'call_input_difference_coordinates.csv', index=False)
unmatched_left_df.to_csv(TABLES / 'unmatched_51r2_calls.csv', index=False)
unmatched_right_df.to_csv(TABLES / 'unmatched_53r2_calls.csv', index=False)

counts = comparison_df.diagnosis.value_counts().to_dict() if len(comparison_df) else {}
all_pairs_exact = bool(len(comparison_df) and (comparison_df.diagnosis == 'CALL_EXACT').all())

if counts.get('OBSERVED_INPUT_DIFFERS', 0):
    decisive_diagnosis = 'OBSERVED_INPUT_DIFFERENCE_FOUND'
elif counts.get('REGIONAL_MASK_DIFFERS', 0):
    decisive_diagnosis = 'REGIONAL_MASK_DIFFERENCE_FOUND'
elif counts.get('INPAINT_PARAMETERS_DIFFER', 0):
    decisive_diagnosis = 'INPAINT_PARAMETER_DIFFERENCE_FOUND'
elif counts.get('IDENTICAL_INPUTS_BUT_RAW_RESULT_DIFFERS', 0):
    decisive_diagnosis = 'RUNTIME_OR_MEMORY_LAYOUT_DIFFERENCE'
elif all_pairs_exact and not unmatched_left and not unmatched_right:
    decisive_diagnosis = 'CALLS_EXACT_DIFFERENCE_IS_POST_INPAINT_COMPOSITION'
else:
    decisive_diagnosis = 'PAIRING_INCOMPLETE_REVIEW_UNMATCHED_CALLS'

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'Fresh 51R2 versus 53R2 cv2.inpaint call-input comparison',
    'paired_calls': int(len(comparison_df)),
    'unmatched_51r2_calls': int(len(unmatched_left)),
    'unmatched_53r2_calls': int(len(unmatched_right)),
    'diagnosis_counts': {str(k): int(v) for k,v in counts.items()},
    'decisive_diagnosis': decisive_diagnosis,
    'expected_historical_stage_f_residual_pixels': EXPECTED_TOTAL_RESIDUAL,
    'stage_f_pass_declared': False,
    'software': {
        'python': sys.version,
        'opencv': cv2.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'platform': platform.platform(),
    },
}
(OUT / 'call_input_comparator_verdict.json').write_text(json.dumps(verdict, indent=2))

print(json.dumps(verdict, indent=2))
print('\nComparison table:')
display(comparison_df.sort_values(['diagnosis','page_51','region_51']))
print('\nOutputs:', OUT)
